In [ ]:
# Confidence-Gated LMS Triage — single-cell Colab notebook
# Classifies student forum doubts by topic (9 Stanford MOOC courses) and by urgency,
# then routes them: auto-handle only when confidently NOT urgent (P(urgent) < 0.15).

import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score

RANDOM_STATE = 42

# --- 1. Load the real data (GitHub, public, no auth) -----------------------
# Each of the 9 courses ships as two partition files (e.g. acc1, acc2) --
# these are two chunks of the SAME course, not two different courses.
BASE = "https://raw.githubusercontent.com/pcla-code/forum-posts-urgency/main/data-per-course/"
COURSE_FILES = [
    "acc1_CODED.csv", "acc2_CODED.csv",
    "calc1_CODED.csv", "calc2_CODED.csv",
    "design1_CODED.csv", "design2_CODED.csv",
    "gam1_CODED.csv", "gam2_CODED.csv",
    "global1_CODED.csv", "global2_CODED.csv",
    "modern1_CODED.csv", "modern2_CODED.csv",
    "mythology1_CODED.csv", "mythology2_CODED.csv",
    "probability1_CODED.csv", "probability2_CODED.csv",
    "vaccines1_CODED.csv", "vaccines2_CODED.csv",
]

frames = []
for fname in COURSE_FILES:
    course = re.sub(r"[12]_CODED\.csv$", "", fname)  # e.g. "acc1_CODED.csv" -> "acc"
    part = pd.read_csv(BASE + fname)
    part["course"] = course
    frames.append(part)

doubts = pd.concat(frames, ignore_index=True)
print("Raw triage data:", doubts.shape)

# --- 2. Clean -----------------------------------------------------------
# `id` is a per-course-local sequential ID, not a global primary key -- the same
# `id` recurs across different courses by coincidence, so a composite key is
# needed before dedup or valid rows get wrongly dropped.
doubts = doubts.dropna(subset=["post_text", "Urgency_1_7"]).copy()
doubts["composite_id"] = doubts["course"] + "_" + doubts["id"].astype(str)
n_before = len(doubts)
doubts = doubts.drop_duplicates(subset="composite_id")
print(f"Dropped {n_before - len(doubts)} true duplicate rows (same course + id)")

doubts = doubts.drop(columns=["id", "composite_id", "post_time"])
doubts["urgent"] = (doubts["Urgency_1_7"] >= 4).astype(int)

print("\nClean shape:", doubts.shape)
print("\nTopic (course) distribution:")
print(doubts["course"].value_counts())
print("\nUrgency label distribution (1 = urgent, Urgency_1_7 >= 4):")
print(doubts["urgent"].value_counts(normalize=True))

# --- 3. Split -------------------------------------------------------------
# Stratify on course + urgent jointly so both splits keep the same topic/urgency balance.
strat_key = doubts["course"] + "_" + doubts["urgent"].astype(str)
train_d, temp_d = train_test_split(doubts, test_size=0.30, stratify=strat_key, random_state=RANDOM_STATE)
strat_key_temp = temp_d["course"] + "_" + temp_d["urgent"].astype(str)
val_d, test_d = train_test_split(temp_d, test_size=0.50, stratify=strat_key_temp, random_state=RANDOM_STATE)
print(f"\ntrain={train_d.shape}, val={val_d.shape}, test={test_d.shape}")

# --- 4. Features ------------------------------------------------------------
vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, stop_words="english")
X_train_txt = vec.fit_transform(train_d["post_text"])
X_val_txt = vec.transform(val_d["post_text"])
X_test_txt = vec.transform(test_d["post_text"])

# --- 5. Topic classifier (9-class) -------------------------------------------
topic_clf = LogisticRegression(max_iter=1000, class_weight="balanced", C=5, random_state=RANDOM_STATE)
topic_clf.fit(X_train_txt, train_d["course"])
val_topic_pred = topic_clf.predict(X_val_txt)
print(f"\nTOPIC — val macro-F1: {f1_score(val_d['course'], val_topic_pred, average='macro'):.3f}")

# --- 6. Urgency classifier, calibrated (probability needs to be trustworthy,
# since the routing decision below depends on it, not just the class label) ---
base_urgency = LogisticRegression(max_iter=1000, class_weight="balanced", C=5, random_state=RANDOM_STATE)
urgency_clf = CalibratedClassifierCV(base_urgency, method="sigmoid", cv=5)
urgency_clf.fit(X_train_txt, train_d["urgent"])

val_proba = urgency_clf.predict_proba(X_val_txt)[:, 1]
val_pred_urgent = (val_proba >= 0.5).astype(int)
print(f"\nURGENCY — val macro-F1 @ 0.5: {f1_score(val_d['urgent'], val_pred_urgent, average='macro'):.3f}")
print(f"URGENCY — val balanced accuracy @ 0.5: {balanced_accuracy_score(val_d['urgent'], val_pred_urgent):.3f}")
print(classification_report(val_d["urgent"], val_pred_urgent))

# --- 7. Routing: auto-handle only when confidently NOT urgent -----------------
# Deliberately asymmetric -- missing a genuinely urgent doubt is far more costly
# than a teacher reviewing one extra doubt that turns out fine. Threshold swept
# on validation, reported once on test.
thresholds = np.arange(0.05, 0.55, 0.05)
rows = []
for t in thresholds:
    auto_mask = val_proba < t
    coverage = auto_mask.mean()
    missed_urgent_rate = val_d["urgent"].values[auto_mask].mean() if auto_mask.sum() > 0 else np.nan
    rows.append({
        "threshold": round(t, 2),
        "auto_handled_pct": round(coverage * 100, 1),
        "urgent_missed_pct_of_auto": round(missed_urgent_rate * 100, 2) if pd.notna(missed_urgent_rate) else None,
    })

threshold_table = pd.DataFrame(rows)
print("\nThreshold sweep (validation):")
print(threshold_table.to_string(index=False))

# --- 8. Final check on test, at the chosen threshold (0.15) -------------------
CHOSEN_THRESHOLD = 0.15
test_proba = urgency_clf.predict_proba(X_test_txt)[:, 1]
auto_mask_test = test_proba < CHOSEN_THRESHOLD

coverage_test = auto_mask_test.mean()
missed_urgent_test = test_d["urgent"].values[auto_mask_test].mean() if auto_mask_test.sum() > 0 else np.nan

print(f"\nTEST — auto-handled: {coverage_test*100:.1f}% of doubts")
print(f"TEST — urgent doubts missed among auto-handled: {missed_urgent_test*100:.2f}%")
print(f"TEST — remaining {100 - coverage_test*100:.1f}% routed to teacher review")
